# IntAct Unsupervised Pre-training - Data Preparation

Using [IntAct dataset](https://www.ebi.ac.uk/intact/download/datasets#mutations) for unsupervised pre-training of StaBddG's folding part.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import gc
import os

import pandas as pd
from sklearn.model_selection import train_test_split

from stabddg.constants import COORDS_ORDER
from stabddg.intact import (
    IntactDataController,
    fetch_alphafold_atoms_parallel,
    fetch_assemblies_atoms_parallel,
    fetch_assemblies_for_uniprots_parallel,
    normalize_entry,
    write_all_safetensors_parallel,
)
from stabddg.intact_pretrain import IntactDataset

# Defines

In [ ]:
url = "https://ftp.ebi.ac.uk/pub/databases/intact/current/various/mutations.tsv"

data_dir = "data"

In [ ]:
structures_dir = os.path.join(data_dir, "intact", "structures")
os.makedirs(structures_dir, exist_ok=True)

# Raw Data

## IntAct Mutations dataset

IntAct Mutations dataset contains info on pairs of proteins: how a mutation in one of the protein impacts the binding of the two proteins.

The dataset does not contain any ∆G measurements per se.

In [ ]:
dc_intact = IntactDataController(
    output_dir=os.path.join(os.path.join(data_dir, "intact"))
)

In [ ]:
dc_intact.prepare_data()

# Protein 3D Structures

AlphaFold only for the time being

In [ ]:
dc_intact.df = pd.read_parquet(os.path.join(data_dir, "intact", "df_intact_mutations.parquet"))

output_dir = os.path.join(structures_dir, "proteins")
os.makedirs(output_dir, exist_ok=True)

## Downloading

In [ ]:
# Collecting all UniProt codes
uniprot_codes = pd.concat(
    [
        dc_intact.df["participant_protein"],
        dc_intact.df["affected_protein_ac"]
    ],
    ignore_index=True
).unique().tolist()

In [ ]:
batch_size = 1000

In [ ]:
df_structures = pd.DataFrame()

In [ ]:
# Iteratively processing
for i in range(0, len(uniprot_codes), batch_size):
    print(f'Batch {i + 1}')
    cur_codes = uniprot_codes[i:(i + batch_size)]

    cur_results = fetch_alphafold_atoms_parallel(
        uniprot_codes=cur_codes,
        max_workers=32
    )

    cur_df_list = []
    for k, v in cur_results.items():
        if v["status"] == "success":
            df_cur = v["df"].copy()
            df_cur["uniprot_code"] = k
            cur_df_list.append(df_cur)
    
    df_structures = pd.concat([df_structures, pd.concat(cur_df_list)], ignore_index=True)
    df_structures.to_parquet(
        os.path.join(dc_intact.output_dir, "df_protein_structures.parquet")
    )

    del cur_results, cur_df_list

In [ ]:
df_counts = df_structures.loc[
    df_structures["atom_name"].isin(["N", "C", "CA"])
].groupby("uniprot_code").size().sort_values()

## Saving as Safetensors

In [ ]:
sidechain_atoms = COORDS_ORDER

max_workers = 6

In [ ]:
# Reading structures data
df_structures = pd.read_parquet(
    os.path.join(dc_intact.output_dir, "df_protein_structures.parquet"),
    columns=["uniprot_code", "res_name_1", "res_name_3", "resnum_label", "resnum_auth", "atom_name", "x", "y", "z"]
)

# Filtering out atoms 
df_structures = df_structures.loc[
    df_structures["atom_name"].isin(sidechain_atoms)
].reset_index(drop=True)

In [ ]:
_ = write_all_safetensors_parallel(
    df_structures=df_structures,
    structures_dir=output_dir,
    max_workers=max_workers,
)

# Assemblies

## Getting All Assemblies

In [ ]:
uniprots_pairs = [
    list(k) 
    for k, v in dc_intact.df.groupby(['participant_protein', 'affected_protein_ac']).groups.items()
]

In [ ]:
df_assemblies_raw = fetch_assemblies_for_uniprots_parallel(
    uniprots_pairs=uniprots_pairs,
    max_workers=32
)

In [ ]:
df_assemblies_raw.to_parquet(os.path.join(data_dir, "intact", "df_assemblies_raw.parquet"))

## Getting All Structures + Writing as Safetensors

In [ ]:
output_dir = os.path.join(structures_dir, "assemblies")
os.makedirs(output_dir, exist_ok=True)

In [ ]:
df_assemblies_raw = pd.read_parquet(
    os.path.join(data_dir, "intact", "df_assemblies_raw.parquet")
)

df_assemblies = df_assemblies_raw.copy().rename(
    columns={"assembly_id": "biological_assembly"}
)

df_assemblies = df_assemblies.loc[
    df_assemblies["biological_assembly"].notna()
].reset_index(drop=True)

df_assemblies = df_assemblies.loc[
    df_assemblies['score'] == df_assemblies.groupby('pair_idx')['score'].transform('min')
].reset_index(drop=True)

df_assemblies["entry_id"] = df_assemblies["biological_assembly"].apply(normalize_entry)

In [ ]:
assembly_ids = df_assemblies["biological_assembly"].unique().tolist()

In [ ]:
_ = fetch_assemblies_atoms_parallel(
    asm_ids=assembly_ids,
    structures_dir=output_dir
)

# Train / Valid / Test split

In [ ]:
valid_size = 0.1
test_size = 0.1
random_state = 42

In [ ]:
feature_type_pos = [
    "mutation causing(MI:2227)",
    "mutation increasing(MI:0382)",
    "mutation increasing rate(MI:1131)",
    "mutation increasing strength(MI:1132)"
]
feature_type_neg = [
    "mutation decreasing(MI:0119)",
    "mutation decreasing rate(MI:1130)",
    "mutation decreasing strength(MI:1133)",
    "mutation disrupting(MI:0573)",
    "mutation disrupting rate(MI:1129)",
    "mutation disrupting strength(MI:1128)",
]

In [ ]:
# Explicitly extracting UniProt codes for further filtering
# TODO There needs to be a better way of capturing which uniprot codes were downloaded
#  without having to reopen the structures dataframe
uniprot_codes = list(df_structures["uniprot_code"].unique())

In [ ]:
df = pd.read_parquet(os.path.join(data_dir, "intact", "df_intact_mutations.parquet"))

In [ ]:
df = df.loc[
    (df["participant_protein"].isin(uniprot_codes))
    & (df["affected_protein_ac"].isin(uniprot_codes))
].reset_index(drop=True)

In [ ]:
# `feature_type_category` column is used for stratification
df["feature_type_category"] = "neutral"
df.loc[df["feature_type"].isin(feature_type_pos), "feature_type_category"] = "positive"
df.loc[df["feature_type"].isin(feature_type_neg), "feature_type_category"] = "negative"

In [ ]:
df_train, df_valid = train_test_split(
    df,
    test_size=(valid_size + test_size),
    stratify=df["feature_type_category"],
    random_state=random_state
)

In [ ]:
df_valid, df_test = train_test_split(
    df_valid,
    test_size=test_size / (valid_size + test_size),
    stratify=df_valid["feature_type_category"],
    random_state=random_state
)

In [ ]:
# Removing the `feature_type_category` column
for df_cur in [df_train, df_valid, df_test]:
    df_cur.drop(columns=["feature_type_category"], inplace=True)
    df_cur.reset_index(drop=True, inplace=True)

In [ ]:
df_train.to_parquet(os.path.join(data_dir, "intact", "df_intact_mutations_filtered_train.parquet"))
df_valid.to_parquet(os.path.join(data_dir, "intact", "df_intact_mutations_filtered_valid.parquet"))
df_test.to_parquet(os.path.join(data_dir, "intact", "df_intact_mutations_filtered_test.parquet"))

In [ ]:
del df_structures
gc.collect()

# Data Loaders

Checking that data loaders work as they should.

In [ ]:
ds_train = IntactDataset(
    df_intact_path=os.path.join(data_dir, "intact", "df_intact_mutations_filtered_train.parquet"),
    structures_dir=structures_dir,
)

In [ ]:
idx = 1000

In [ ]:
%%timeit
tst = ds_train[idx]